In [ ]:
import mlflow
import matplotlib.pyplot as plt
import geopandas as gpd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearnex import patch_sklearn
from sklearn.manifold import TSNE
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

patch_sklearn()

In [ ]:
import numpy as np

num_features = 10
patch_size = 5

np.random.normal(loc=0, scale=1e-2, size=(num_features)).astype(np.half)

In [ ]:
np.array([ 0.004402  , -0.0009613 ,  0.0122    , -0.005836  ,  0.001667  ,
        0.002434  , -0.00399   ,  0.0002251 , -0.00010735, -0.010735  ], dtype=np.half)

In [ ]:
EXPERIMENT_NAME = "brazil-finetuning"

In [ ]:
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
runs_df.set_index("tags.mlflow.runName", inplace=True)

In [ ]:
RUN_NAME = "BERTSCRATCH-70"

In [ ]:
test_gdf = gpd.read_parquet(f"{runs_df.loc[RUN_NAME].artifact_uri}/test.parquet")
train_gdf = gpd.read_parquet(f"{runs_df.loc[RUN_NAME].artifact_uri}/train.parquet")
val_gdf = gpd.read_parquet(f"{runs_df.loc[RUN_NAME].artifact_uri}/val.parquet")

In [ ]:
CLASS_NAMES = sorted(test_gdf.y_true.unique().tolist())
X_COLUMNS = [col for col in train_gdf.columns if col.startswith("emb_")]

In [ ]:
def plot_tsne_from_gdf(gdf, color_column, n_samples=1000, ax=None, color_map=None, alpha=0.9):
    n_samples = min(n_samples, len(gdf))
    sample_gdf = gdf.sample(n=n_samples, random_state=42)
    
    X_sample = sample_gdf[X_COLUMNS]

    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_jobs=-1)
    X_tsne = tsne.fit_transform(X_sample)

    sample_gdf["tsne1"] = X_tsne[:, 0]
    sample_gdf["tsne2"] = X_tsne[:, 1]
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
        
    palette_to_use = color_map if color_map is not None else 'deep'

    sns.scatterplot(
        data=sample_gdf, 
        x='tsne1', 
        y='tsne2', 
        hue=color_column,
        palette=palette_to_use,
        alpha=alpha,
        s=50,
        ax=ax
    )

    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

    ax.legend(title=color_column.replace("_", " ").title(), bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, linestyle='--', alpha=0.6)
    
    if ax is None:
        plt.tight_layout()
        plt.show()

In [ ]:
fixed_crop_colors = {
    'Other Temporary Crops': '#9A6324',
    'Pasture': '#469990',
    'Sugar Cane': '#FFD700',
    'Coffee': '#4363d8',
    'Forest Plantation': '#3cb44b',
    'Other Perennial Crops': '#f58231',
    'Soybean': '#ffe119',
    'Citrus': '#f032e6',
    'Palm Oil': '#800000',
    'Rice': '#000075',
    'ZUnknow': '#000000'
}

In [ ]:
train_gdf["gmm_pred"] = train_gdf.apply(lambda row: "ZUnknow" if row.gmm_gemos_anomaly else row.y_pred, axis=1)
val_gdf["gmm_pred"] = val_gdf.apply(lambda row: "ZUnknow" if row.gmm_gemos_anomaly else row.y_pred, axis=1)
test_gdf["gmm_pred"] = test_gdf.apply(lambda row: "ZUnknow" if row.gmm_gemos_anomaly else row.y_pred, axis=1)

In [ ]:
fig, ax = plt.subplots(1, 4, sharex=True, sharey=True, figsize=(32, 7))

plot_tsne_from_gdf(test_gdf, color_column="y_true", color_map=fixed_crop_colors, n_samples=3000, ax=ax[0], alpha=0.7)
plot_tsne_from_gdf(test_gdf, color_column="gmm_pred", color_map=fixed_crop_colors, n_samples=3000, ax=ax[1], alpha=0.7)
plot_tsne_from_gdf(test_gdf[~test_gdf.gmm_gemos_anomaly], color_column="y_true", color_map=fixed_crop_colors, n_samples=3000, ax=ax[2], alpha=0.7)
plot_tsne_from_gdf(test_gdf[~test_gdf.gmm_gemos_anomaly], color_column="gmm_pred", color_map=fixed_crop_colors, n_samples=3000, ax=ax[3], alpha=0.7)

ax[0].set_title("True Classes", fontsize=16)
ax[1].set_title("Predicted Classes", fontsize=16)
ax[2].set_title("True Classes (Without Anomalies)", fontsize=16)
ax[3].set_title("Predicted Classes (Without Anomalies)", fontsize=16)

ax[0].legend_.remove()
ax[1].legend_.remove()
ax[2].legend_.remove()

plt.savefig(f"figs/{RUN_NAME}_TSNE_embeddings_test_set.pdf", bbox_inches='tight')

In [ ]:
def plot_classification_results(y_true, y_pred, filename):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    class_names = sorted(list(set(y_true) | set(y_pred)))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax1,
        xticklabels=class_names,
        yticklabels=class_names,
    )

    ax1.set_xlabel("Predicted")
    ax1.set_ylabel("True")

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=[str(c) for c in class_names],
        output_dict=True,
        zero_division=1,
    )
    report_df = pd.DataFrame(report_dict).T

    to_drop = ["accuracy", "macro avg", "weighted avg"]
    report_df = report_df.drop(index=[idx for idx in to_drop if idx in report_df.index])

    report_df[["precision", "recall", "f1-score"]] = (
        report_df[["precision", "recall", "f1-score"]] * 100
    ).round()
    report_df = report_df.astype(int)

    sns.heatmap(report_df, annot=True, cmap="RdYlGn", ax=ax2, fmt="d", vmin=0, vmax=100)

    plt.savefig(filename, bbox_inches='tight')

In [ ]:
plot_classification_results(val_gdf.y_true, val_gdf.y_pred, f"figs/{RUN_NAME}_classification_report_val_set.pdf")
plot_classification_results(test_gdf.y_true, test_gdf.y_pred, f"figs/{RUN_NAME}_classification_report_test_set.pdf")

plot_classification_results(val_gdf.y_true, val_gdf.gmm_pred, f"figs/{RUN_NAME}_classification_report_val_set_gemos.pdf")
plot_classification_results(test_gdf.y_true, test_gdf.gmm_pred, f"figs/{RUN_NAME}_classification_report_test_set_gemos.pdf")

In [ ]:
gdf = test_gdf

In [ ]:
gdf.gmm_score

In [ ]:
fig, ax = plt.subplots(3, len(CLASS_NAMES), figsize=(25, 7))

for n, crop_class in enumerate(CLASS_NAMES):
    subpart_gdf = gdf[gdf.y_pred==crop_class].reset_index(drop=True)

    subpart_gdf[subpart_gdf.right_pred].y_proba.plot.hist(color="green", ax=ax[0][n])
    subpart_gdf[~subpart_gdf.right_pred].y_proba.plot.hist(color="red", ax=ax[0][n], alpha=0.5)
    
    ax[0][n].set_title(crop_class)
    ax[1][n].set_title("")
    ax[2][n].set_title("")

    subpart_gdf[subpart_gdf.right_pred].y_conf.plot.hist(color="green", ax=ax[1][n])
    subpart_gdf[~subpart_gdf.right_pred].y_conf.plot.hist(color="red", ax=ax[1][n], alpha=0.5)

    subpart_gdf[subpart_gdf.right_pred].gmm_score.plot.hist(color="green", ax=ax[2][n])
    subpart_gdf[~subpart_gdf.right_pred].gmm_score.plot.hist(color="red", ax=ax[2][n], alpha=0.5)
    
    # Add ylabel only for the first column
    if n == 0:
        ax[0][n].set_ylabel("MCP")
        ax[1][n].set_ylabel("ConfidNet")
        ax[2][n].set_ylabel("GMM")
    else:
        ax[0][n].set_ylabel("")
        ax[1][n].set_ylabel("")
        ax[2][n].set_ylabel("")

plt.savefig(f"figs/{RUN_NAME}_uncertainty_histograms_test_set.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(2, gdf.crop_class.nunique(), sharex="col", sharey="col", figsize=(25, 7))

for n, crop_class in enumerate(sorted(gdf.crop_class.unique().tolist())):
    subpart_gdf = gdf[gdf.crop_class==crop_class].reset_index(drop=True)

    subpart_gdf[subpart_gdf.is_correct].pred_proba.plot.hist(color="green", ax=ax[1][n])
    subpart_gdf[~subpart_gdf.is_correct].pred_proba.plot.hist(color="red", ax=ax[1][n], alpha=0.5)
    ax[0][n].set_title(crop_class + " - MCP")

    subpart_gdf[subpart_gdf.is_correct].pred_conf.plot.hist(color="green", ax=ax[0][n])
    subpart_gdf[~subpart_gdf.is_correct].pred_conf.plot.hist(color="red", ax=ax[0][n], alpha=0.5)
    ax[1][n].set_title("Confidnet")


plt.tight_layout()
plt.show()